In [ ]:
%%bash

docker compose -f /home/brijeshdhaker/IdeaProjects/bd-notebooks-module/docker-compose.yml down minio

In [ ]:
%%bash

docker compose -f /home/brijeshdhaker/IdeaProjects/bd-notebooks-module/docker-compose.yml up -d minio


In [ ]:
import os
from pyspark import SparkConf
from pyspark.sql import SparkSession
from pyspark.sql.functions import *
from pyspark.sql.types import *
from delta import *

# Adding AWS S3 Minio configs
sparkConf = (
    SparkConf()
    .set("spark.jars.ivy","/home/brijeshdhaker/.ivy2")
    .set("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension")
    .set("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog")
    .set("spark.jars.packages","org.apache.hadoop:hadoop-aws:3.3.4,com.amazonaws:aws-java-sdk-bundle:1.12.797,io.delta:delta-spark_2.12:3.3.2")
    .set("spark.executor.heartbeatInterval", "300000")
    .set("spark.network.timeout", "400000")
    .set("spark.hadoop.fs.defaultFS", "s3a://defaultfs/")
    .set("spark.hadoop.fs.s3a.endpoint", "http://minio.sandbox.net:9010")
    .set("spark.hadoop.fs.s3a.access.key", "pgm2H2bR7a5kMc5XCYdO")
    .set("spark.hadoop.fs.s3a.secret.key", "zjd8T0hXFGtfemVQ6AH3yBAPASJNXNbVSx5iddqG")
    .set("spark.hadoop.fs.s3a.path.style.access", "true")
    .set("spark.hadoop.fs.s3a.aws.credentials.provider", "org.apache.hadoop.fs.s3a.SimpleAWSCredentialsProvider")
    .set("spark.hadoop.fs.s3a.impl", "org.apache.hadoop.fs.s3a.S3AFileSystem")
    .set("spark.hadoop.delta.enableFastS3AListFrom", "true")
    #.set("spark.eventLog.enabled", "true")
    #.set("spark.eventLog.dir", "file:///apps/var/logs/spark-events")
)

# configure the SparkSession with the configure_spark_with_delta_pip() utility function in Delta Lake:
builder = SparkSession.builder.appName("spark-deltalake").master("local[*]").config(conf=sparkConf)
spark = configure_spark_with_delta_pip(builder, extra_packages=["org.apache.hadoop:hadoop-aws:3.3.4"]).getOrCreate()

#
spark.sparkContext.setLogLevel('ERROR')
spark

#
# 
#
def display(df):
    df.show(truncate=False)

#### Delete Existing Delta Table

In [ ]:
%%bash

## Delete Existing Delta Table
aws --endpoint-url http://minio.sandbox.net:9010 s3 rm s3://defaultfs/deltalake/peoples --recursive


#### Create Deltatable

In [ ]:
from pyspark.sql.types import StructType, StructField, IntegerType, StringType, TimestampType

schema = StructType([
  StructField("id", IntegerType(), True),
  StructField("firstName", StringType(), True),
  StructField("middleName", StringType(), True),
  StructField("lastName", StringType(), True),
  StructField("gender", StringType(), True),
  StructField("birthDate", TimestampType(), True),
  StructField("ssn", StringType(), True),
  StructField("salary", IntegerType(), True)
])

df = spark.read.format("csv").option("header", True).schema(schema).load("s3a://datasets/people-data/peoples.csv")
df.printSchema()

#
display(df)

In [ ]:
%load_ext sql

In [ ]:
%sql spark

#### Create Delta table

In [ ]:
# Save as delta table into Minio S3
df.write.format('delta').save('/deltalake/spark_df_peoples')

# Create the table if it does not exist. Otherwise, replace the existing table.
#df.writeTo("spark_catalog.default.peoples").createOrReplace()

# If you know the table does not already exist, you can call this instead:
#df.write.saveAsTable("spark_catalog.default.peoples")

#### Read a Deltatable

In [ ]:
spark_df_peoples = spark.read.format('delta').load("/deltalake/spark_df_peoples")
display(spark_df_peoples)

In [ ]:
%%sql

DESCRIBE HISTORY delta.`s3a://defaultfs/deltalake/spark_df_peoples`

In [ ]:
from delta.tables import *

deltaTable = DeltaTable.forPath(spark, 's3a://defaultfs/deltalake/spark_df_peoples')
display(deltaTable.history())

In [ ]:
%%sql

INSERT INTO  delta.`s3a://defaultfs/deltalake/spark_df_peoples`
SELECT * FROM PARQUET.`s3a://defaultfs/deltalake/spark_df_peoples`;

In [ ]:
%%sql

SELECT MIN(salary), MAX(salary), COUNT(id)
FROM delta.`s3a://defaultfs/deltalake/spark_df_peoples`

In [ ]:
%%sql 

UPDATE delta.`s3a://defaultfs/deltalake/spark_df_peoples`
SET salary = 6523
WHERE salary = -6523;

In [ ]:
%%sql

DESCRIBE HISTORY delta.`s3a://defaultfs/deltalake/spark_df_peoples`

In [ ]:
%%sql 

SELECT *
FROM delta.`s3a://defaultfs/deltalake/spark_df_peoples`
WHERE id = 1

In [ ]:
%%sql 

DELETE
FROM delta.`s3a://defaultfs/deltalake/spark_df_peoples`
WHERE id = 1

In [ ]:
%%sql 

SELECT *
FROM delta.`s3a://defaultfs/deltalake/spark_df_peoples`
WHERE id = 1

In [ ]:
%%sql

DESCRIBE HISTORY delta.`s3a://defaultfs/deltalake/spark_df_peoples`

In [ ]:
for i in range(5):
    spark.sql(
        """
        INSERT INTO delta.`s3a://defaultfs/deltalake/spark_df_peoples`
        SELECT * FROM PARQUET.`s3a://defaultfs/deltalake/spark_df_peoples`;
        """
    )
    print(f"{i}th INSERT completed ")

In [ ]:
%%sql

DESCRIBE HISTORY delta.`s3a://defaultfs/deltalake/spark_df_peoples`

#### Python Create Deltatable

In [ ]:
spark.sql("""

CREATE OR REPLACE TABLE delta.`s3a://defaultfs/deltalake/people10m` (
  id INT,
  firstName STRING,
  middleName STRING,
  lastName STRING,
  gender STRING,
  birthDate TIMESTAMP,
  ssn STRING,
  salary INT
) USING DELTA
          
""")

In [ ]:
from delta import *

DeltaTable.createIfNotExists(spark).location("s3a://defaultfs/deltalake/people00m") \
    .tableName("people00m") \
    .addColumn("id", "INT") \
    .addColumn("firstName", "STRING") \
    .addColumn("middleName", "STRING") \
    .addColumn("lastName", "STRING", comment = "surname") \
    .addColumn("gender", "STRING") \
    .addColumn("birthDate", "TIMESTAMP") \
    .addColumn("ssn", "STRING") \
    .addColumn("salary", "INT") \
    .execute()

In [ ]:
from delta import *
from pyspark.sql import SparkSession
from pyspark.sql.functions import *

deltaTable = DeltaTable.forPath(spark, 's3a://defaultfs/deltalake/people00m')
deltaTable.toDF().show()


In [ ]:
%%sql

DESCRIBE HISTORY delta.`s3a://defaultfs/deltalake/people00m`

In [ ]:
%%sql 

DESCRIBE HISTORY default.people12m;

In [ ]:
%load_ext sql

In [ ]:
%%sql spark

select * from people00m;

In [ ]:
%%sql

USE default;

In [ ]:
%%sql 

show tables;

In [ ]:
%%sql 

DESCRIBE default.people12m

In [ ]:
%%sql 

DESCRIBE EXTENDED default.people12m

In [ ]:
%%sql

CREATE OR REPLACE TABLE delta.`s3a://defaultfs/deltalake/people11m` (
  id INT,
  firstName STRING,
  middleName STRING,
  lastName STRING,
  gender STRING,
  birthDate TIMESTAMP,
  ssn STRING,
  salary INT
) USING DELTA


In [ ]:
%%sql 

CREATE TABLE default.people12m (
    id INT,
    firstName STRING,
    middleName STRING,
    lastName STRING,
    gender STRING,
    birthDate TIMESTAMP,
    ssn STRING,
    salary INT
)
USING DELTA
LOCATION 's3a://defaultfs/deltalake/people12m'

In [ ]:
%%sql

CREATE DATABASE IF NOT EXISTS nyc;

In [ ]:
%%sql

USE nyc;

In [ ]:
%%sql 

show tables;

#### Upsert to a Deltatable

In [ ]:
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, DateType
from datetime import date

schema = StructType([
  StructField("id", IntegerType(), True),
  StructField("firstName", StringType(), True),
  StructField("middleName", StringType(), True),
  StructField("lastName", StringType(), True),
  StructField("gender", StringType(), True),
  StructField("birthDate", DateType(), True),
  StructField("ssn", StringType(), True),
  StructField("salary", IntegerType(), True)
])

data = [
  (9999998, 'Billy', 'Tommie', 'Luppitt', 'M', date.fromisoformat('1992-09-17'), '953-38-9452', 55250),
  (9999999, 'Elias', 'Cyril', 'Leadbetter', 'M', date.fromisoformat('1984-05-22'), '906-51-2137', 48500),
  (10000000, 'Joshua', 'Chas', 'Broggio', 'M', date.fromisoformat('1968-07-22'), '988-61-6247', 90000),
  (20000001, 'John', '', 'Doe', 'M', date.fromisoformat('1978-01-14'), '345-67-8901', 55500),
  (20000002, 'Mary', '', 'Smith', 'F', date.fromisoformat('1982-10-29'), '456-78-9012', 98250),
  (20000003, 'Jane', '', 'Doe', 'F', date.fromisoformat('1981-06-25'), '567-89-0123', 89900)
]

people_10m_updates = spark.createDataFrame(data, schema)
people_10m_updates.createOrReplaceTempView("people_10m_updates")

# ...

from delta.tables import DeltaTable

deltaTable = DeltaTable.forPath(spark, '/deltalake/peoples')

(deltaTable.alias("people_10m")
  .merge(
    people_10m_updates.alias("people_10m_updates"),
    "people_10m.id = people_10m_updates.id"
  )
  .whenMatchedUpdateAll()
  .whenNotMatchedInsertAll()
  .execute()
)

#### Read a Deltatable

In [ ]:
df = spark.read.format('delta').load("/deltalake/peoples")
df_filtered = df.filter(df["id"] >= 9999998)
display(df_filtered)

#### Write to a Deltatable

In [ ]:
# df.write.mode("append").saveAsTable("main.default.people_10m")

# Save as delta table
df.write.format('delta').mode('append').save('/deltalake/delta-table')

In [ ]:
# df.write.mode("overwrite").saveAsTable("main.default.people_10m")

# Save as delta table
df.write.format('delta').mode('overwrite').save('/deltalake/delta-table')

#### Update Deltatable Rows

In [ ]:
from delta import *
from pyspark.sql import SparkSession
from pyspark.sql.functions import *

deltaTable = DeltaTable.forPath(spark, '/deltalake/peoples')

# Declare the predicate by using a SQL-formatted string.
deltaTable.update(
  condition = "gender = 'F'",
  set = { "gender": "'Female'" }
)

# Declare the predicate by using Spark SQL functions.
deltaTable.update(
  condition = col('gender') == 'M',
  set = { 'gender': lit('Male') }
)

#### Delete Rows 

In [ ]:
from delta.tables import *
from pyspark.sql.functions import *

deltaTable = DeltaTable.forPath(spark, '/deltalake/peoples')

# Declare the predicate by using a SQL-formatted string.
deltaTable.delete("birthDate < '1955-01-01'")

# Declare the predicate by using Spark SQL functions.
deltaTable.delete(col('birthDate') < '1960-01-01')

#### Display table history

In [ ]:
from delta.tables import *

deltaTable = DeltaTable.forPath(spark, '/deltalake/peoples')
display(deltaTable.history())

#### overwrite

In [ ]:
# Save as delta table
df.write.format('delta').mode('overwrite').save('/deltalake/delta-table')

#### Time Travle

In [ ]:
# Read version 1
from delta.tables import *

deltaTable = DeltaTable.forPath(spark, '/deltalake/peoples')
deltaHistory = deltaTable.history()

display(deltaHistory.where("version == 0"))
# Or:
display(deltaHistory.where("timestamp == '2024-05-15T22:43:15.000+00:00'"))

In [ ]:
df = spark.read.format('delta').option('versionAsOf', 0).load("/deltalake/peoples")
# Or: 2025-10-14 18:39:41
#df = spark.read.format('delta').option('timestampAsOf', '2025-10-14T18:45:03.000+00:00').load("/deltalake/peoples")

display(df)

#### Display table history

In [ ]:
deltaTable = DeltaTable.forPath(spark, "/deltalake/peoples")
print("######## Describe history for the table ######")
deltaTable.history().show()

#### Vacuum

In [ ]:
deltaTable = DeltaTable.forPath(spark, "/deltalake/peoples")
print("######## Vacuum the table ########")
deltaTable.vacuum()

#### Describe details for the table

In [ ]:
print("######## Describe details for the table ######")
deltaTable.detail().show()

#### Generating manifest 

In [ ]:
# Generate manifest
print("######## Generating manifest ######")
deltaTable.generate("SYMLINK_FORMAT_MANIFEST")

In [ ]:
# SQL Vacuum
print("####### SQL Vacuum #######")
spark.sql("VACUUM '%s' RETAIN 169 HOURS" % ("/deltalake/peoples")).collect()

In [ ]:
# SQL describe history
print("####### SQL Describe History ########")
print(spark.sql("DESCRIBE HISTORY delta.`%s`" % ("/deltalake/peoples")).collect())

In [ ]:
import shutil

# cleanup
shutil.rmtree("/tmp/delta-table")

In [ ]:
%%bash

aws --endpoint-url http://minio.sandbox.net:9010 s3 rm s3://defaultfs/deltalake/peoples --recursive

#### Optimize a table
After you have performed multiple changes to a table, you might have a lot of small files. To improve the speed of read queries, you can use the optimize operation to collapse small files into larger ones:

In [ ]:
from delta.tables import *

#deltaTable = DeltaTable.forName(spark, "main.default.people_10m")
deltaTable = DeltaTable.forPath(spark, "/deltalake/peoples")
deltaTable.optimize().executeCompaction()

#### Z-order by columns
To improve read performance further, you can collocate related information in the same set of files by z-ordering. Delta Lake data-skipping algorithms use this collocation to dramatically reduce the amount of data that needs to be read. To z-order data, you specify the columns to order on in the z-order by operation. For example, to collocate by gender, run:

In [ ]:
from delta.tables import *

#deltaTable = DeltaTable.forName(spark, "main.default.people_10m")
deltaTable = DeltaTable.forPath(spark, "/deltalake/peoples")
deltaTable.optimize().executeZOrderBy("gender")

#### Clean up snapshots with VACUUM
Delta Lake provides snapshot isolation for reads, which means that it is safe to run an optimize operation even while other users or jobs are querying the table. Eventually however, you should clean up old snapshots. You can do this by running the vacuum operation:

In [ ]:
from delta.tables import *

#deltaTable = DeltaTable.forName(spark, "main.default.people_10m")
deltaTable = DeltaTable.forPath(spark, "/deltalake/peoples")
deltaTable.vacuum()

#### How do I find the last commit's version in the Spark session?

In [ ]:
spark.conf.get("spark.databricks.delta.lastCommitVersionInSession")